# 05 · Tabu search

Simulated annealing escapes local optima by **luck**: it randomly accepts worse moves, less and less over time. **Tabu search** escapes by **memory**:

1. Every step, look at a set of possible moves and make the **best** one, even if it's worse than now.
2. Remember what was just changed and **forbid undoing it** for a few steps (the *tabu list*), so the search can't walk straight back up the hill it just left.
3. Keep the best schedule ever seen.

Picture walking down from a hilltop in fog, with a rule: *you may not step on the last 7 squares you stood on*. You're forced onwards, over the next ridge, towards new ground.

Tabu search was introduced by Fred Glover in 1986 and is widely used for scheduling and routing. Like hill climbing and SA, it starts from an **existing** schedule.

All the code in this notebook is complete. Read it, run it, and change the settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research').is_dir())
sys.path.insert(0, str(ROOT))

import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from research.evaluate import score, summary, validate
from research.local_search import Schedule
from research.reference.greedy import run_greedy
from research.reference.hill_climbing import MOVE_WEIGHTS, MOVES, hill_climb
from research.reference.simulated_annealing import simulated_annealing
from research.synthetic import Scenario, generate

data = generate(seed=1)
greedy_result = run_greedy(**data)
print('Greedy score:', score(greedy_result, data).total)

## 1. The four ingredients

**a. A sampled neighbourhood.** In principle tabu search looks at *every* possible move each step. Here that's thousands (every task × every shift × every technician swap), so each step **samples** `NEIGHBOURS_PER_STEP` random moves and scores them all. Sampling is the standard trick for big problems.

**b. Take the best move, even if it's worse.** Hill climbing only moves uphill and SA *sometimes* accepts a worse move by chance. Tabu search **always moves**, to the best option on offer. At a local optimum every option is worse, so it steps down to the *least bad* one, on purpose.

**c. The tabu list.** Without memory, the next step would just undo that step down (it's the best move back up), and the search would bounce between two schedules forever. So after moving a task, that **task is tabu for `TABU_TENURE` steps**: no move may touch it. Storing a simple *attribute* of a move (here: which task), rather than whole schedules, is how tabu lists are usually done; it's cheap and works well.

**d. The aspiration rule.** A tabu move is still allowed if it would give the **best schedule ever seen**. The tabu list exists to stop cycling, not to block a genuine new record.

## 2. The tabu search loop

The same building blocks as notebooks 03 and 04 (`Schedule`, the moves, `apply`/`undo`), arranged differently:

- **Inner loop:** try `NEIGHBOURS_PER_STEP` moves. Apply each, score it, **undo it right away**, and remember the best allowed one.
- **Then** make that best move for real and mark its task tabu.

In [ ]:
NEIGHBOURS_PER_STEP = 20
TABU_TENURE = 7
TIME_LIMIT_S = 3.0


def my_tabu_search(data, start_result, neighbours=NEIGHBOURS_PER_STEP, tenure=TABU_TENURE,
                   time_limit_s=TIME_LIMIT_S, seed=0):
    rng = random.Random(seed)
    names = list(MOVES)
    weights = [MOVE_WEIGHTS[name] for name in names]

    schedule = Schedule(data, start_result)
    current = schedule.total()
    best, best_snapshot = current, schedule.snapshot()
    tabu_until = {}                                   # task_id -> last step it is still tabu
    trace = {'current': [], 'best': [], 'moved_task': [], 'went_down': [], 'aspiration': []}
    started, step = time.perf_counter(), 0

    while time.perf_counter() - started < time_limit_s:
        # a. look at a sample of moves and remember the best allowed one
        chosen = None                                 # (score, change, aspiration)
        for _ in range(neighbours):
            name = rng.choices(names, weights=weights)[0]
            change = MOVES[name](schedule, rng)
            if change is None or not schedule.apply(change):
                continue                              # breaks a hard rule
            candidate = schedule.total()
            schedule.undo(change)                     # only looking, not moving yet

            is_tabu = tabu_until.get(change.task_id, -1) >= step
            aspiration = is_tabu and candidate > best  # d. a new record beats the tabu rule
            if is_tabu and not aspiration:
                continue                              # c. tabu: not allowed
            if chosen is None or candidate > chosen[0]:
                chosen = (candidate, change, aspiration)

        # b. make the best move, even if it is worse than now
        moved_task, went_down, aspiration = None, False, False
        if chosen is not None and schedule.apply(chosen[1]):
            candidate, change, aspiration = chosen
            went_down = candidate < current
            current = candidate
            tabu_until[change.task_id] = step + tenure  # c. this task is now tabu
            moved_task = change.task_id
            if current > best:
                best, best_snapshot = current, schedule.snapshot()

        trace['current'].append(current)
        trace['best'].append(best)
        trace['moved_task'].append(moved_task)
        trace['went_down'].append(went_down)
        trace['aspiration'].append(aspiration)
        step += 1

    schedule.restore(best_snapshot)
    return schedule, trace


started = time.perf_counter()
schedule, trace = my_tabu_search(data, greedy_result)
result = schedule.to_result('tabu_search', started)

hc_history = hill_climb(data, greedy_result)[1]
sa_trace = simulated_annealing(data, greedy_result, time_limit_s=TIME_LIMIT_S)[1]
print('Problems:', validate(result, data) or 'none')
print(f"Greedy {score(greedy_result, data).total}   hill climbing {hc_history[-1]}   "
      f"simulated annealing {sa_trace['best'][-1]}   tabu search {score(result, data).total}")
print(f"Tabu: {len(trace['current'])} steps x {NEIGHBOURS_PER_STEP} moves looked at;  "
      f"SA: {len(sa_trace['current'])} single tries, both in {TIME_LIMIT_S} s")

Notice the last line: in the same time, tabu search makes **far fewer steps** than SA makes tries, because every step scores 20 moves. Each step is more expensive but better informed. That's the basic trade-off between the two.

## 3. Watching it work

1. **Current vs best score.** After the first climb, the current score keeps stepping **down** (the least bad move) and back up somewhere new, while the best only rises. Compared with SA, the movement is steadier: tabu chooses, SA rolls dice.
2. **Which task moved at each step.** Each dot is one step. Because a moved task is tabu for `TABU_TENURE` steps, the same task never reappears within that gap, unless the step beat the best ever (red: the aspiration rule).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(trace['current'], label='current', alpha=0.8)
axes[0].plot(trace['best'], label='best so far', linewidth=2)
axes[0].axhline(hc_history[-1], color='grey', linestyle='--', label='hill climbing finished here')
axes[0].set_ylabel('score')
axes[0].legend()

steps = [i for i, task in enumerate(trace['moved_task']) if task is not None]
tasks = [trace['moved_task'][i] for i in steps]
colors = ['#dc2626' if trace['aspiration'][i] else ('#f59e0b' if trace['went_down'][i] else '#10b981') for i in steps]
axes[1].scatter(steps, tasks, c=colors, s=12)
axes[1].set_ylabel('task moved')
axes[1].set_xlabel('step')
axes[1].set_title('green = went up, orange = went down (least bad move), red = tabu but allowed (new best)', fontsize=9)
plt.tight_layout()
plt.show()

print(f"Steps that went down on purpose: {sum(trace['went_down'])} of {len(trace['went_down'])}")
print(f"Aspiration (tabu move allowed because it set a new best): {sum(trace['aspiration'])}")

In [ ]:
hc_result = hill_climb(data, greedy_result)[0].to_result('hill_climbing', 0.0)
sa_result = simulated_annealing(data, greedy_result, time_limit_s=TIME_LIMIT_S)[0].to_result('sa', 0.0)
pd.DataFrame([summary(greedy_result, data), summary(hc_result, data), summary(sa_result, data), summary(result, data)],
             index=['greedy', 'hill climbing', 'simulated annealing', 'tabu search']).drop(columns='runtime_ms')

## 4. Why the tabu list matters

Run the same search with `tenure=0` (no memory). It takes the least bad move down, then the best move straight back up, and bounces between the same few schedules. With memory it keeps exploring.

In [ ]:
plt.figure(figsize=(10, 3.5))
for tenure in [0, 3, 7, 15]:
    run = my_tabu_search(data, greedy_result, tenure=tenure, time_limit_s=1.5)[1]
    plt.plot(run['best'], label=f"tenure {tenure}: best {run['best'][-1]:.0f}")
plt.xlabel('step')
plt.ylabel('best score so far')
plt.legend()
plt.show()

## 5. Five runs: how consistent is it?

Five random seeds on the same problem, 1.5 seconds each, tabu search vs simulated annealing.

In [ ]:
rows = []
for run_seed in range(5):
    sa_final = simulated_annealing(data, greedy_result, time_limit_s=1.5, seed=run_seed)[1]['best'][-1]
    tabu_final = my_tabu_search(data, greedy_result, time_limit_s=1.5, seed=run_seed)[1]['best'][-1]
    rows.append({'seed': run_seed, 'simulated annealing': sa_final, 'tabu search': tabu_final})
five = pd.DataFrame(rows)
pd.concat([five, five[['simulated annealing', 'tabu search']].agg(['mean', 'min', 'max', 'std']).round(1)])

## 6. All four methods, twenty problems, same time

Each improvement method gets the same budget per problem. Hill climbing restarts with new seeds to use its time. This cell takes about a minute.

In [ ]:
BUDGET_S = 1.0


def hill_climb_with_restarts(data, start_result, time_limit_s):
    started, best_score, run_seed = time.perf_counter(), None, 0
    while time.perf_counter() - started < time_limit_s:
        final = hill_climb(data, start_result, seed=run_seed, max_stall=500)[1][-1]
        best_score = final if best_score is None else max(best_score, final)
        run_seed += 1
    return best_score


rows = []
for seed in range(20):
    problem = generate(seed=seed)
    start = run_greedy(**problem)
    tabu_schedule, tabu_trace = my_tabu_search(problem, start, time_limit_s=BUDGET_S)
    rows.append({
        'seed': seed,
        'greedy': score(start, problem).total,
        'hill climbing': hill_climb_with_restarts(problem, start, BUDGET_S),
        'simulated annealing': simulated_annealing(problem, start, time_limit_s=BUDGET_S)[1]['best'][-1],
        'tabu search': tabu_trace['best'][-1],
        'tabu valid': not validate(tabu_schedule.to_result('tabu', 0.0), problem),
    })
comparison = pd.DataFrame(rows)
methods = ['greedy', 'hill climbing', 'simulated annealing', 'tabu search']
comparison['winner'] = comparison[methods].idxmax(axis=1)
comparison

In [ ]:
print('Wins per method:', comparison['winner'].value_counts().to_dict())
gains = comparison[methods[1:]].sub(comparison['greedy'], axis=0)
gains.describe().round(1).rename(index={'mean': 'mean gain over greedy'})

## 7. What you can tune

| Setting | What it does | Too high | Too low |
|:---|:---|:---|:---|
| `TABU_TENURE` | How long a moved task stays forbidden | Too many tasks blocked; the best moves are often forbidden | Cycles back to schedules it just left (0 = no memory) |
| `NEIGHBOURS_PER_STEP` | How many moves are scored per step | Each step is slow, so few steps fit in the budget | Steps are poorly informed, close to random |
| What is tabu | Here: the task. Alternatives: the exact (task, team, day) it left, or the technician | Strict attributes block too much | Loose attributes still allow cycles |
| Aspiration rule | When a tabu move is allowed anyway | | Without it, record-breaking moves can be blocked |
| Budget | How long it searches | Diminishing returns | Stops before exploring |

A common rule of thumb: tenure around 5–15, or a small fraction of the number of things that can be tabu (here 25 tasks). These can be tuned with Optuna exactly as in notebook 04.

Change the values below and run the cell.

In [ ]:
settings = {'neighbours': 20, 'tenure': 7, 'time_limit_s': 3.0}

tuned_schedule, tuned_trace = my_tabu_search(data, greedy_result, **settings)
print(settings, '->', tuned_trace['best'][-1], f"({len(tuned_trace['current'])} steps)")
plt.figure(figsize=(9, 3))
plt.plot(tuned_trace['current'], alpha=0.8, label='current')
plt.plot(tuned_trace['best'], linewidth=2, label='best so far')
plt.legend()
plt.show()

## 8. Simulated annealing vs tabu search

| | Simulated annealing | Tabu search |
|:---|:---|:---|
| How it escapes local optima | **Luck**: accepts a worse move with probability `e^(Δ/T)` | **Memory**: always makes the best allowed move; recent changes are tabu |
| Moves looked at per step | 1 | Many (a sample of the neighbourhood) |
| Steps per second | Many, cheap | Few, expensive |
| Main settings | Start/end temperature, cooling, budget | Tenure, neighbours per step, what is tabu, budget |
| Behaviour over time | Changes: explores early, polishes late | Steady: same rules every step |
| Randomness | High (random moves + random acceptance) | Lower (random sample, but a deterministic choice) |
| Easy to add to hill climbing? | Very: change the accept rule | Moderate: add sampling, a tabu list and aspiration |

**Which one wins depends on the problem**, which is why you measure both on the same problems and the same time budget, as above. Many production solvers offer both, plus *late acceptance*, and let you pick or tune.

## Summary

- Tabu search = every step, make the **best allowed move**, even a worse one; recently moved things are **tabu** for a few steps.
- The tabu list stops the search from cycling straight back; the **aspiration rule** lets a new record through anyway.
- Big neighbourhoods are **sampled**, not searched completely.
- Keep the **best schedule seen**; compare with **the same time budget**.

The full notes, including a comparison of all four methods, are in `notes/personal_notes.md`.